In [ ]:
import json
import requests
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
# 1) Cargar API key desde el JSON
with open("../config/api_config.json", "r", encoding="utf-8") as f:
    api_key = json.load(f)["aemet_api_key"]

In [ ]:
# 2) Endpoint que quieres consultar
municipio_id = "01051" 
url_api = f"https://opendata.aemet.es/opendata/api/prediccion/especifica/municipio/diaria/{municipio_id}"

In [ ]:
# 3) Primera llamada: obtiene la URL "datos"
resp = requests.get(url_api, params={"api_key": api_key}, timeout=30)
resp.raise_for_status()
meta = resp.json()

# AEMET suele devolver: estado, descripcion, datos, metadatos...
if "datos" not in meta:
    raise RuntimeError(f"No se encontró 'datos' en la respuesta: {meta}")

datos_url = meta["datos"]

In [ ]:

# 4) Segunda llamada: descarga los datos reales (robusta)

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session.mount("https://", HTTPAdapter(max_retries=retries))

headers = {
    "Accept": "application/json",
    "User-Agent": "python-requests aemet-client"
}

resp_datos = session.get(datos_url, headers=headers, timeout=30)
resp_datos.raise_for_status()

try:
    prediccion = resp_datos.json()
except ValueError:
    prediccion = resp_datos.text


In [ ]:

root = prediccion[0]
municipio_id = str(root.get("id", "")).zfill(5)   # 1051 -> "01051"
elaborado = root.get("elaborado")
dias = root["prediccion"]["dia"]

# -------------------------
# 1) prediccion_diaria
# -------------------------
rows_diaria = []
for d in dias:
    rows_diaria.append({
        "municipio_id": municipio_id,
        "fecha": pd.to_datetime(d["fecha"]).date(),
        "elaborado": elaborado,
        "temp_max": d.get("temperatura", {}).get("maxima"),
        "temp_min": d.get("temperatura", {}).get("minima"),
        "hum_max": d.get("humedadRelativa", {}).get("maxima"),
        "hum_min": d.get("humedadRelativa", {}).get("minima"),
        "uv_max": d.get("uvMax"),
    })

df_diaria = pd.DataFrame(rows_diaria)

# -------------------------
# 2) prediccion_por_periodo
# (probPrecipitacion, estadoCielo, viento, cotaNieveProv, rachaMax)
# -------------------------
periodo_keys = ["probPrecipitacion", "cotaNieveProv", "estadoCielo", "viento", "rachaMax"]

rows_periodo = []
for d in dias:
    fecha = pd.to_datetime(d["fecha"]).date()
    for k in periodo_keys:
        for item in d.get(k, []):
            rows_periodo.append({
                "municipio_id": municipio_id,
                "fecha": fecha,
                "tipo": k,
                "periodo": item.get("periodo"),        # a veces no viene
                "value": item.get("value"),
                "descripcion": item.get("descripcion"),
                "direccion": item.get("direccion"),
                "velocidad": item.get("velocidad"),
            })

df_periodo = pd.DataFrame(rows_periodo)

# -------------------------
# 3) prediccion_por_hora
# (temperatura.dato, humedadRelativa.dato, sensTermica.dato)
# -------------------------
hora_keys = ["temperatura", "humedadRelativa", "sensTermica"]

rows_hora = []
for d in dias:
    fecha = pd.to_datetime(d["fecha"]).date()
    for k in hora_keys:
        for item in d.get(k, {}).get("dato", []):
            rows_hora.append({
                "municipio_id": municipio_id,
                "fecha": fecha,
                "tipo": k,
                "hora": item.get("hora"),
                "value": item.get("value"),
            })

df_hora = pd.DataFrame(rows_hora)



In [ ]:
provincia = root.get("provincia")
municipio_nombre = root.get("nombre")

In [ ]:
provincia, municipio_nombre

In [ ]:
df_diaria

In [ ]:
df_periodo.head(40)

In [ ]:
df_hora